# Notebook 3: Khởi tạo **AFK-MC2** và phân cụm **K-means**

## Đọc nhanh kết quả Notebook 2 (AIC / BIC & PCA)

**Biểu đồ AIC / BIC theo K (GMM)**  
- Cả AIC và BIC **giảm mạnh** khi tăng số thành phần — mô hình khớp dữ liệu hơn nhưng phức tạp hơn.  
- Có **khúc gập rõ** khoảng **K = 3 → 4** và **K = 5 → 6**: sau đó độ giảm **chậm dần** (kiểu “elbow”).  
- Trong dải bạn vẽ, BIC thường **thấp nhất** về phía **K lớn** (ví dụ 10–12); tại **K = 11** có nhích **lên** một chút so với K = 10 — gợi ý thêm thành phần không còn “đáng” với mức phạt BIC.  

**Ý cho đề tài:** số thành phần **GMM** (mô hình hỗn hợp) **không bắt buộc** trùng số cụm **K-means** dùng cho phân khúc khách. Thường chọn **K cụm marketing nhỏ hơn** (4–6) để dễ diễn giải, dù BIC GMM thích K lớn hơn.

**Biểu đồ PCA–2D (nhãn GMM)**  
- Hai PC giải thích phần lớn phương sai → hình chiếu **đại diện tốt** cho RFM đã chuẩn hóa.  
- Có **chùm tách khá rõ** (ví dụ một dải gần tuyến tính) và vùng **chồng lấn ở giữa** — bình thường với dữ liệu thật và với nhiều thành phần GMM.

---

## Pipeline **đúng tinh thần paper** (GMS → AFK-MC2 trên tập con → K-means full)

Trong bài gốc, thứ tự là:

1. **Gms:** học **GMM** trên toàn bộ dữ liệu, **sinh mẫu** từ GMM, rồi với mỗi mẫu tìm **điểm thật gần nhất** trong tập gốc → được **tập con** `DS_sample`.  
2. **AFK-MC2** chạy trên **`DS_sample`** để có bộ tâm khởi tạo tốt.  
3. Bước cuối trên **toàn bộ dữ liệu**: paper gọi tiếp một lần **AFK-MC2** với tâm đã có; package Python `afkmc2` **chỉ** có hàm `afkmc2(X, k)` **không** nhận tâm sẵn cho lần hai.  
4. **Cách làm tương đương thực tế (đã ghi trong báo cáo):** dùng **K-means (Elkan)** trên **toàn bộ `X`** với `init=` đúng **k tâm** vừa tính từ AFK-MC2 trên tập con — tức **tinh chỉnh Lloyd** từ seed paper-style.

Ngoài ra notebook vẫn giữ baseline **AFK-MC2 trực tiếp trên full `X`** để so sánh với pipeline trên.

## Mục tiêu Notebook 3

1. Cài **Gms** (lấy mẫu + map về điểm thật) và **AFK-MC2 trên tập con** → **K-means full**.  
2. So sánh với **AFK-MC2 trên full**, **K-means++**, **random init** (SSE, Silhouette, Davies–Bouldin).  
3. Lưu nhãn cụm cho bước phân tích RFM.

**Cài đặt:** `python3 -m pip install -r requirements.txt` (cần `afkmc2`; GMM dùng `sklearn`).

In [ ]:
from __future__ import annotations

from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.mixture import GaussianMixture

try:
    import afkmc2.afkmc2 as afk_seed
except ImportError as e:
    raise ImportError(
        "Cần cài afkmc2: python3 -m pip install afkmc2"
    ) from e

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "rfm_customers.csv"
OUT_DIR = ROOT / "data"
MODEL_PATH = ROOT / "models" / "gmm_rfm.joblib"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = ["R_z", "F_z", "M_z"]
RANDOM_STATE = 42

# Số cụm K-means (phân khúc)
N_CLUSTERS = 4

# --- GMS (giống paper): tỷ lệ lấy mẫu trên n khách
SAMPLE_RATE = 0.5

# Trọng số từng chiều khi tìm điểm thật gần nhất (paper: ω). [1,1,1] = Euclidean thông thường trên (R_z,F_z,M_z)
OMEGA = np.ones(len(FEATURE_COLS), dtype=float)

# GMM cho GMS: ưu tiên load `models/gmm_rfm.joblib` từ NB2; không có thì fit mới
GMM_FALLBACK_N_COMPONENTS = 8
GMM_N_INIT = 5
GMM_MAX_ITER = 200
GMM_REG_COVAR = 1e-3

AFKMC2_CHAIN_LEN = 200
KMEANS_MAX_ITER = 300

In [ ]:
rfm = pd.read_csv(DATA_PATH)
missing = [c for c in FEATURE_COLS if c not in rfm.columns]
if missing:
    raise ValueError(f"Thiếu cột {missing}. Chạy 01_DataProcessing.ipynb trước.")

X = rfm[FEATURE_COLS].to_numpy(dtype=float)
ids = rfm["customer_id"].to_numpy()
print("X shape:", X.shape, "| k =", N_CLUSTERS)

In [ ]:
np.random.seed(RANDOM_STATE)


def gms_nearest_real_points(
    X_data: np.ndarray,
    gmm: GaussianMixture,
    n_sample: int,
    omega: np.ndarray,
) -> np.ndarray:
    """Gms (theo paper): sinh n_sample điểm từ GMM, mỗi điểm map sang hàng trong X_data gần nhất (khoảng cách có trọng số omega)."""
    synth, _ = gmm.sample(n_sample)
    omega = np.asarray(omega, dtype=float).reshape(1, -1)
    idx = []
    for s in synth:
        diff = X_data - s
        dist = np.linalg.norm(diff * omega, axis=1)
        idx.append(int(np.argmin(dist)))
    return X_data[np.array(idx, dtype=int)]


def load_or_fit_gmm_for_gms(X_data: np.ndarray) -> GaussianMixture:
    if MODEL_PATH.is_file():
        bundle = joblib.load(MODEL_PATH)
        gmm = bundle["gmm"]
        return gmm
    gmm = GaussianMixture(
        n_components=GMM_FALLBACK_N_COMPONENTS,
        covariance_type="full",
        random_state=RANDOM_STATE,
        n_init=GMM_N_INIT,
        max_iter=GMM_MAX_ITER,
        reg_covar=GMM_REG_COVAR,
    )
    gmm.fit(X_data)
    return gmm


n = X.shape[0]
n_sample = max(N_CLUSTERS, int(round(SAMPLE_RATE * n)))
gmm_gms = load_or_fit_gmm_for_gms(X)
X_sub = gms_nearest_real_points(X, gmm_gms, n_sample, OMEGA)
print(f"GMS: n_sample={n_sample} (rate≈{SAMPLE_RATE}), X_sub shape={X_sub.shape}")

# --- Pipeline paper-style: AFK-MC2 chỉ trên tập con → seed cho K-means full
init_from_sub = afk_seed.afkmc2(X_sub, N_CLUSTERS, m=AFKMC2_CHAIN_LEN)
km_paper = KMeans(
    n_clusters=N_CLUSTERS,
    init=init_from_sub,
    n_init=1,
    max_iter=KMEANS_MAX_ITER,
    random_state=RANDOM_STATE,
    algorithm="elkan",
)
km_paper.fit(X)
print("Pipeline GMS + AFK-MC2(sub) + K-means(full) — SSE:", km_paper.inertia_, "| n_iter:", km_paper.n_iter_)

# --- Baseline: AFK-MC2 trực tiếp trên toàn bộ X (không qua GMS / tập con)
init_full = afk_seed.afkmc2(X, N_CLUSTERS, m=AFKMC2_CHAIN_LEN)
km_afk_full = KMeans(
    n_clusters=N_CLUSTERS,
    init=init_full,
    n_init=1,
    max_iter=KMEANS_MAX_ITER,
    random_state=RANDOM_STATE,
    algorithm="elkan",
)
km_afk_full.fit(X)
print("Baseline AFK-MC2(full) + K-means — SSE:", km_afk_full.inertia_, "| n_iter:", km_afk_full.n_iter_)

In [ ]:
km_pp = KMeans(
    n_clusters=N_CLUSTERS,
    init="k-means++",
    n_init=10,
    max_iter=KMEANS_MAX_ITER,
    random_state=RANDOM_STATE,
    algorithm="elkan",
)
km_pp.fit(X)

km_rand = KMeans(
    n_clusters=N_CLUSTERS,
    init="random",
    n_init=10,
    max_iter=KMEANS_MAX_ITER,
    random_state=RANDOM_STATE,
    algorithm="elkan",
)
km_rand.fit(X)

print("K-means++ inertia:", km_pp.inertia_)
print("K-means random inertia:", km_rand.inertia_)

In [ ]:
def metrics_for_model(name: str, model: KMeans) -> dict:
    """SSE (trong sklearn) = inertia = tổng bình phương khoảng cách tới tâm cụm được gán."""
    lab = model.labels_
    return {
        "method": name,
        "SSE_inertia": float(model.inertia_),
        "silhouette": float(silhouette_score(X, lab)),
        "davies_bouldin": float(davies_bouldin_score(X, lab)),
        "n_iter": int(model.n_iter_),
    }


rows = [
    metrics_for_model("GMS + AFK-MC2 (sub) + K-means (full)", km_paper),
    metrics_for_model("AFK-MC2 (full) + K-means", km_afk_full),
    metrics_for_model("K-means++", km_pp),
    metrics_for_model("K-means (random init)", km_rand),
]
metrics_df = pd.DataFrame(rows)
metrics_df

In [ ]:
# Gộp nhãn cụm (chính: AFK-MC2 + K-means) vào bảng RFM gốc để phân tích sau
out = rfm.copy()
out["kmeans_gms_afkmc2_pipeline"] = km_paper.labels_
out["kmeans_afkmc2_full"] = km_afk_full.labels_
out["kmeans_pp"] = km_pp.labels_
out["kmeans_random"] = km_rand.labels_

out_path = OUT_DIR / "rfm_with_kmeans_clusters.csv"
out.to_csv(out_path, index=False)
print("Đã lưu:", out_path.resolve())

metrics_path = OUT_DIR / "kmeans_init_comparison.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Đã lưu:", metrics_path.resolve())
out.head()

In [ ]:
# So sánh trực quan (PCA-2D)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X2 = pca.fit_transform(X)

fig, axes = plt.subplots(2, 2, figsize=(9, 8), sharex=True, sharey=True)
axes = axes.ravel()
titles_labels = [
    ("GMS + AFK-MC2(sub) + K(full)", km_paper.labels_),
    ("AFK-MC2(full) + K", km_afk_full.labels_),
    ("K-means++", km_pp.labels_),
    ("Random init", km_rand.labels_),
]
for ax, (title, lab) in zip(axes, titles_labels):
    ax.scatter(X2[:, 0], X2[:, 1], c=lab, cmap="tab10", s=10, alpha=0.55)
    ax.set_title(title)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[2].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.suptitle("So sánh nhãn cụm (chiếu PCA-2D)", y=1.01)
plt.tight_layout()
plt.show()